In [3]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential

In [5]:
# 1. Cargar métricas e hiperparámetros guardados
import json

with open('../outputs/metrics.json') as f:
    metrics = json.load(f)

with open ('../outputs/hyperparams.json') as f:
    hyperparams = json.load(f)

In [6]:
# 2. Conectar a ws
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [7]:
print(ml_client.workspace_name)

mlw-churn-dev


In [14]:
# 3. Registrar modelo desde carpeta local
model = Model(
    path='../outputs/model.pkl',
    type=AssetTypes.CUSTOM_MODEL,
    name='telco-churn-model',
    # version=2,
    description='Telco customer churn prediction - HistGradientBoosting',
    tags={
        # Framework
        'framework': 'scikit-learn',
        'algorithm': 'HistGradientBoosting',

        # Metricas
        'accuracy': f"{metrics['accuracy']:.4f}",
        'precision': f"{metrics['precision']:.4f}",
        'recall': f"{metrics['recall']:.4f}",
        'f1_score': f"{metrics['f1_score']:.4f}",

        # Hiperparametros clave
        'learning_rate': str(hyperparams['learning_rate']),
        'max_depth': str(hyperparams['max_depth']),
        'max_iter': str(hyperparams['max_iter']),
        'class_weight': 'balanced',

        # Configuración
        'threshold': '0.3', 

        #
        'stage': 'candidate',
    }
)

registered_model = ml_client.models.create_or_update(model)

print(f"Modelo registrado: {registered_model.name}, versión: {registered_model.version}")

Uploading model.pkl (< 1 MB): 100%|██████████| 725k/725k [00:00<00:00, 15.1MB/s]




Modelo registrado: telco-churn-model, versión: 3


In [8]:
# Registrar modelo desde ultimo job

# 1. Obtener jobs y encontrar el último
jobs = list(ml_client.jobs.list())

# Filtrar por experiment name
churn_jobs = [j for j in jobs if j.experiment_name == "churn-prediction-training"]

# Ordenar por fecha (más reciente primero)
churn_jobs.sort(key=lambda x: x.creation_context.created_at, reverse=True)

latest_job = churn_jobs[0]
print(f"Job más reciente: {latest_job.name}, Status: {latest_job.status}")

# 2. Registrar modelo DESDE ese job
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model = Model(
    path=f"azureml://jobs/{latest_job.name}/outputs/artifacts/paths/outputs/model.pkl",
    type=AssetTypes.CUSTOM_MODEL,
    name='telco-churn-model',
    description='Churn model WITHOUT FeatureAdder',
    tags={
        'framework': 'scikit-learn',
        'no_custom_transformers': 'true',
        'class_weight': 'balanced',
        'stage': 'production',
    }
)

registered_model = ml_client.models.create_or_update(model)
print(f"Registrado: {registered_model.name} v{registered_model.version}")

Job más reciente: gentle_frame_ckh7d79mc6, Status: Completed
Registrado: telco-churn-model v5


In [10]:
model.tags['threshold'] = 'pending_analysis'
ml_client.models.create_or_update(registered_model)

Model({'job_name': 'gentle_frame_ckh7d79mc6', 'intellectual_property': None, 'system_metadata': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-model', 'description': 'Churn model WITHOUT FeatureAdder', 'tags': {'framework': 'scikit-learn', 'no_custom_transformers': 'true', 'class_weight': 'balanced', 'stage': 'production', 'threshold': 'pending_analysis'}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/9cbbe496-fe29-459a-a3d7-44790ebac1fb/resourceGroups/rg-churn-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-churn-dev/models/telco-churn-model/versions/5', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-churn-dev/code/Users/asandovh/telco-churn/notebooks', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7855a34787f0>, 'serialize': <msrest.serialization.Serializer object at 0x7855a3280310>, 'version': '5', 'latest_ve